# Question 3:
The main shortcoming of basic BOW is that it weights all words equally, meaning common stopwords are treated as important as emotionally significant words like "scared" or "happy." These stopwords add noise without providing any discriminative signal for classification.

# Question 8(1):
P@1,100 measures retrieval accuracy: the model has to pick the correct response from 100 candidates. AVG-BLEU is the average of BLEU-1 thru BLEU-4.Checks how much the model response overlaps. PPL is only used for generative models and shows how well the model predicts the gold response: lower perplexity means the model assigns a higher probability to the correct output.

P@1,100 is the most useful metric for empathy systems because it is not just matching exact wording. A response can still be empathetic without sharing many words with a gold answer. This makes BLEU and perplexity less reliable.

# Question 8(2):
Fine-tuning based on the empathetic dialogue data has a chance to greatly improve empathy scores with minimal cost. However the gold standard is still far from these models currently. More premium models may score better but would require retraining, further finetuned data and significant costs.

# Generative AI Attestation

The use of generative AI in this assignment was limited and primarily contained within the Google Colab environment. I used Google Colab’s autocomplete feature when I was stuck on specific Python syntax, and occasionally consulted its built-in AI tools to help debug code written outside the provided skeleton. All such instances are documented where applicable. Some incidental exposure to AI-generated content may have occurred during general web searches (e.g., Google Gemini), but it was not directly relied upon for solutions. All ideas, problem-solving approaches, and implementation strategies were independently developed and executed by me.

In [4]:
# 95-891 Introduction to AI Homework 4: NLP
# Brendan Reed - brreed@andrew.cmu.edu

import os
import re

import pandas as pd
import numpy as np

import gensim                   # Run 'pip install gensim' for this import.
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score

import collections
from itertools import compress

# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
data_dir = '/content/drive/MyDrive/AI_NLP'

# Question 1: Data ETL
# ----------------------------------------------------------------

# Importing the datasets
train = pd.read_csv(f'{data_dir}/train.csv', on_bad_lines='skip')
valid = pd.read_csv(f'{data_dir}/valid.csv', on_bad_lines='skip')
test  = pd.read_csv(f'{data_dir}/test.csv',  on_bad_lines='skip')

# 1a: Filter for target sentiments
target_sentiments = {'sad', 'jealous', 'joyful', 'terrified'}
train_full = pd.concat([train, valid], ignore_index=True)

# 1b: Extract utterance (X) and context (y) using pandas.loc
X_train = train_full.loc[train_full['context'].isin(target_sentiments), 'utterance']
y_train = train_full.loc[train_full['context'].isin(target_sentiments), 'context']
X_test  = test.loc[test['context'].isin(target_sentiments), 'utterance']
y_test  = test.loc[test['context'].isin(target_sentiments), 'context']

# clean underscore-encoded spaces in utterances
X_train = X_train.str.replace('_', ' ', regex=False).reset_index(drop=True)
X_test  = X_test.str.replace('_', ' ', regex=False).reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

# Encode string to labels integers
labels_unique   = sorted(target_sentiments)                                 # alphabetical: jealous, joyful, sad, terrified
train_labels_encoded  = np.array([label_mapper[l] for l in y_train])
labels_encoded_test   = np.array([label_mapper[l] for l in y_test])

print(f"X_train : {X_train.shape}")
print(f"X_test : {X_test.shape}")
print(f"Label counts  :\n{y_train.value_counts()}")
print(f"Training label distribution:\n{y_train.value_counts()}\n")


# Text clean. Had to research to complete this function efficiently. Google Colab helped troubleshoot the syntax to this function.
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)   # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Question 2: Bag-of-Words
# -------------------------------------------------------------

traindata_sanitized = [clean_text(sent) for sent in X_train]
testdata_sanitized  = [clean_text(sent) for sent in X_test]

# BOW vocabulary
bow_vector = CountVectorizer()
x_bow = bow_vector.fit_transform(traindata_sanitized)
encoding = x_bow.toarray().astype(float)

# count of words to binary
for arr in encoding:
    arr[arr > 0] = 1


# Question 3: Remove stopwords
# ------------------------------------------------------------

stopwords_list = list(set(stopwords.words('english')))

stopwords_list.extend(['comma', '', 'im', 'ive', 'id', 'dont', 'didnt',         # Extended list of stop words compiled during data exploration.
                       'wasnt', 'isnt', 'cant', 'wouldnt', 'couldnt'])

def remove_stopwords(text: str, sw: list) -> str:                               # Google Colab helped write this function.
    tokens = text.split()                                                       # I had written a function which removed stop words but it was not working properly and began breaking strings.
    tokens = [t for t in tokens if t not in sw]                                 # This function splits utterances into tokens and removes stopwords as tokens instead of string.
    return ' '.join(tokens)

train_nostopwords = [remove_stopwords(s, stopwords_list) for s in traindata_sanitized]
test_nostopwords  = [remove_stopwords(s, stopwords_list) for s in testdata_sanitized]

# Rebuild BOW
train_count_vector = CountVectorizer()
X_train_bow = train_count_vector.fit_transform(train_nostopwords)
train_encoded = X_train_bow.toarray().astype(float)

for arr in train_encoded:
    arr[arr > 0] = 1


# Question 4: Normalization
# ------------------------------------------------------------

train_transformer = TfidfTransformer(smooth_idf=True, use_idf=True)
train_embed_transform = train_transformer.fit_transform(train_encoded)


# Question 5: SGD Classifier
# ------------------------------------------------------------

X_train_sgd = train_embed_transform
y_train_sgd = train_labels_encoded

clf = SGDClassifier(
    loss='modified_huber',   # robust to outliers
    max_iter=200,
    random_state=42
)
clf.fit(X_train_sgd, y_train_sgd)

# Training accuracy
train_sgc = clf.predict(X_train_sgd)
print(f"Q5: Train accuracy : {accuracy_score(y_train_sgd, train_sgc):.4f}")             # Use sklearn accuracy_score

# Prepare test features using training vocab
test_count_vector = CountVectorizer(vocabulary=train_count_vector.vocabulary_)
X_test_bow = test_count_vector.fit_transform(test_nostopwords)
test_encoding = X_test_bow.toarray().astype(float)

for arr in test_encoding:
    arr[arr > 0] = 1

# Normalizing the test data
test_transformer = TfidfTransformer(smooth_idf=False, use_idf=True)
test_embed_transform = test_transformer.fit_transform(test_encoding)

# Getting predictions on test data
test_predicted_labels = clf.predict(test_embed_transform)

# Error Analysis:
test_acc   = accuracy_score(labels_encoded_test, test_predicted_labels)
f1_scores  = f1_score(labels_encoded_test, test_predicted_labels, average=None)
conf_mat   = confusion_matrix(labels_encoded_test, test_predicted_labels)

print(f"Q5: Test accuracy: {test_acc:.4f}")
print(f"Q5: Mean F1: {np.mean(f1_scores):.4f}")
print(f"Q5: Confusion matrix:\n{conf_mat}\n")

# Misclassified
print("\n Q5: Misclassified utterances:")
misclassified_utr = np.where(test_predicted_labels != labels_encoded_test)[0]       # Identify predicted vs encoded mismatch.
inv_mapper        = {v: k for k, v in label_mapper.items()}                         # Google Colab helped write this line.
for i in misclassified_utr[:5]:
    print(f"Utterance  : {X_test.iloc[i]}")                                         # Print examples
    print(f"True label : {inv_mapper[labels_encoded_test[i]]}")
    print(f"Predicted  : {inv_mapper[test_predicted_labels[i]]}\n")

# Question 6: Word2Vec + MLP Classifier
# ------------------------------------------------------------

# Tokenize stopword removed sentences
train_tokens = [nltk.word_tokenize(s) for s in train_nostopwords]
test_tokens  = [nltk.word_tokenize(s) for s in test_nostopwords]

# Load pretrained Word2Vec model
model = gensim.models.KeyedVectors.load_word2vec_format(
    f'{data_dir}/GoogleNews-vectors-negative300.bin', binary=True                       # Google Drive location.
)

def sentence_to_w2v(tokens: list, model, vector_size: int = 300) -> np.ndarray:         # Google Colab helped build these two lines of word2vec function which will be used on train and test data.
    """Average word vectors for all tokens present in the model vocabulary."""
    vectors = [model[token] for token in tokens if token in model]
    return np.mean(vectors, axis=0) if vectors else np.zeros(vector_size)               # I took over builiding this W2V from here, down.

train_w2v = np.array([sentence_to_w2v(t, model) for t in train_tokens])
test_w2v  = np.array([sentence_to_w2v(t, model) for t in test_tokens])

# MLP classifier
mlp_w2v = MLPClassifier(
    activation='relu',
    max_iter=100,
    random_state=42
)
mlp_w2v.fit(train_w2v, train_labels_encoded)                                        # Google Colab autocompleted this line.

test_pred_w2v = mlp_w2v.predict(test_w2v)
acc_w2v = accuracy_score(labels_encoded_test, test_pred_w2v)
f1_w2v = f1_score(labels_encoded_test, test_pred_w2v, average=None)
conf_mat_w2v = confusion_matrix(labels_encoded_test, test_pred_w2v)

print(f"\n Q6: Word2Vec + MLP accuracy : {acc_w2v:.4f}")
print(f"Q6: F1 per class : { {labels_unique[i]: round(f1_w2v[i], 4) for i in range(len(labels_unique))} }")
print(f"Q6: Mean F1 : {np.mean(f1_w2v):.4f}")
print(f"Q6: Confusion matrix :\n{conf_mat_w2v}")


# Question 7: DistilBERT + MLP
# ------------------------------------------------------------

import torch
from transformers import DistilBertTokenizer, DistilBertModel

tokenizer  = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
bert_model = DistilBertModel.from_pretrained('distilbert-base-uncased')
bert_model.eval()

# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model.to(device)

def get_bert_embeddings(texts: list, tokenizer, model, batch_size: int = 32) -> np.ndarray:         # I began building this function and Google Colab helped autocomplete it with appropriate arguments.
    all_embeddings = []                                                                             # Gen AI also helped troubleshoot this line when I was receiving an error. The fix ended up being the line 'batch =' and 'return_tensors='
    for i in range(0, len(texts), batch_size):
        batch   = texts[i : i + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt'
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}                             # Gen AI helped troublshoot this line when I was getting a nondescript error from the console output.
        with torch.no_grad():
            output = model(**encoded)

        cls_vecs = output.last_hidden_state[:, 0, :].cpu().numpy()                          # Google Colab helped autocomplete this line
        all_embeddings.append(cls_vecs)
    return np.vstack(all_embeddings)

# Use the stopword removed text as input to BERT
print("Q7: Encoding training set with DistilBERT")
train_bert = get_bert_embeddings(train_nostopwords, tokenizer, bert_model)
print("Q7: Encoding test set with DistilBERT")
test_bert  = get_bert_embeddings(test_nostopwords,  tokenizer, bert_model)

mlp_bert = MLPClassifier(
    activation='relu',
    max_iter=100,
    random_state=42
)
mlp_bert.fit(train_bert, train_labels_encoded)

test_pred_bert = mlp_bert.predict(test_bert)
acc_bert = accuracy_score(labels_encoded_test, test_pred_bert)
f1_bert = f1_score(labels_encoded_test, test_pred_bert, average=None)
conf_mat_bert = confusion_matrix(labels_encoded_test, test_pred_bert)

print(f"\n Q7: DistilBERT + MLP Test accuracy : {acc_bert:.4f}")
print(f"Q7: F1 per class : { {labels_unique[i]: round(f1_bert[i], 4) for i in range(len(labels_unique))} }")
print(f"Q7: Mean F1 : {np.mean(f1_bert):.4f}")
print(f"Q7: Confusion matrix :\n{conf_mat_bert}")

# Summary table                                                                         # I employed Google Colab Gen AI to wrap this up and compare the models in a readable format.
print("\n" + "="*55)
print("  MODEL COMPARISON SUMMARY")
print("="*55)
print(f"  {'Model':<28} {'Accuracy':>10}  {'Mean F1':>10}")
print("-"*55)
print(f"  {'SGD (TF-IDF BOW)':<28} {test_acc:>10.4f}  {np.mean(f1_scores):>10.4f}")
print(f"  {'Word2Vec + MLP':<28} {acc_w2v:>10.4f}  {np.mean(f1_w2v):>10.4f}")
print(f"  {'DistilBERT + MLP':<28} {acc_bert:>10.4f}  {np.mean(f1_bert):>10.4f}")
print("="*55)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Mounted at /content/drive
X_train : (10609,)
X_test : (720,)
Label counts  :
context
sad          2863
terrified    2677
joyful       2538
jealous      2531
Name: count, dtype: int64
Training label distribution:
context
sad          2863
terrified    2677
joyful       2538
jealous      2531
Name: count, dtype: int64

Q5: Train accuracy : 0.8748


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:1675: RuntimeWarning: divide by zero encountered in divide
  self.idf_ /= df


Q5: Test accuracy: 0.6347
Q5: Mean F1: 0.6361
Q5: Confusion matrix:
[[106  29  31  17]
 [ 17 117  33  20]
 [ 19  25 121  30]
 [ 12  11  19 113]]


 Q5: Misclassified utterances:
Utterance  : she was born premature at home comma  she had hard time breathing on her own but instead of taking her to the doctor parents were just praying
True label : sad
Predicted  : terrified

Utterance  : yes! And i do believe in God and prayers but goodness gracious please take your children to the hospital and let God heal them THROUGH doctors
True label : sad
Predicted  : terrified

Utterance  : One of the saddest things to me is when people underestimate what they can do and/or are capable of.
True label : sad
Predicted  : jealous

Utterance  : That's perfectly natural. You sound like the kind of person comma  though comma  that quickly regained their bearings. Disappointments like that usually come with some *good* lessons.
True label : sad
Predicted  : joyful

Utterance  : I met up with an old flame 

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(



 Q6: Word2Vec + MLP accuracy : 0.6514
Q6: F1 per class : {'jealous': np.float64(0.6534), 'joyful': np.float64(0.5789), 'sad': np.float64(0.6489), 'terrified': np.float64(0.7349)}
Q6: Mean F1 : 0.6541
Q6: Confusion matrix :
[[115  40  14  14]
 [ 20 110  34  23]
 [ 25  30 122  18]
 [  9  13  11 122]]


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q7: Encoding training set with DistilBERT
Q7: Encoding test set with DistilBERT

 Q7: DistilBERT + MLP Test accuracy : 0.6069
Q7: F1 per class : {'jealous': np.float64(0.5448), 'joyful': np.float64(0.5859), 'sad': np.float64(0.6074), 'terrified': np.float64(0.6819)}
Q7: Mean F1 : 0.6050
Q7: Confusion matrix :
[[ 79  48  37  19]
 [ 12 116  37  22]
 [ 10  28 123  34]
 [  6  17  13 119]]

  MODEL COMPARISON SUMMARY
  Model                          Accuracy     Mean F1
-------------------------------------------------------
  SGD (TF-IDF BOW)                 0.6347      0.6361
  Word2Vec + MLP                   0.6514      0.6541
  DistilBERT + MLP                 0.6069      0.6050


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
